In [5]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

api.dataset_download_files(
    "ankitbansal06/retail-orders",
    path="data",
    unzip=True
)

print("Dataset downloaded successfully")


Dataset URL: https://www.kaggle.com/datasets/ankitbansal06/retail-orders
Dataset downloaded successfully


In [12]:
import pandas as pd

df = pd.read_csv("data/orders.csv",na_values=['Not Available','unknown'])




In [14]:
df.head(100)

,Order Id,Order Date,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub Category,Product Id,cost price,List Price,Quantity,Discount Percent
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,2023-12-17,Standard Class,Home Office,United States,Portland,Oregon,97206,West,Office Supplies,Binders,OFF-BI-10004738,10,10,1,2
96,97,2022-04-28,Second Class,Home Office,United States,New York City,New York,10009,East,Furniture,Furnishings,FUR-FU-10000629,80,100,7,5
97,98,2023-06-14,First Class,Consumer,United States,San Francisco,California,94122,West,Office Supplies,Binders,OFF-BI-10001721,40,50,3,4
98,99,2023-02-05,Standard Class,Corporate,United States,Saint Paul,Minnesota,55106,Central,Office Supplies,Appliances,OFF-AP-10000358,70,80,6,3


In [16]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')




In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          9994 non-null   int64  
 1   order_date        9994 non-null   object 
 2   ship_mode         9988 non-null   object 
 3   segment           9994 non-null   object 
 4   country           9994 non-null   object 
 5   city              9994 non-null   object 
 6   state             9994 non-null   object 
 7   postal_code       9994 non-null   int64  
 8   region            9994 non-null   object 
 9   category          9994 non-null   object 
 10  sub_category      9994 non-null   object 
 11  product_id        9994 non-null   object 
 12  cost_price        9994 non-null   int64  
 13  list_price        9994 non-null   int64  
 14  quantity          9994 non-null   int64  
 15  discount_percent  9994 non-null   int64  
 16  discount          9994 non-null   float64


In [17]:
df['discount'] = df['list_price'] * df['discount_percent'] * 0.01
df['sale_price'] = df['list_price'] - df['discount']
df['profit'] = df['sale_price'] - df['cost_price']


In [19]:
df['order_date'] = pd.to_datetime(df['order_date'], format="%Y-%m-%d")


In [20]:
import pyodbc


server = "DESKTOP-K4EF3Q0"
username = "sa"
password = "shah123"

conn = pyodbc.connect(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};DATABASE=master;UID={username};PWD={password}",
    autocommit=True
)
cursor = conn.cursor()
print("Connected to SQL Server!")


Connected to SQL Server!


In [21]:
db_name = "kaggleDataSet"
cursor.execute(f"SELECT name FROM sys.databases WHERE name='{db_name}'")
if not cursor.fetchone():
    cursor.execute(f"CREATE DATABASE {db_name}")
    print(f"Database '{db_name}' created!")
else:
    print(f"Database '{db_name}' already exists.")


Database 'kaggleDataSet' created!


In [22]:
conn = pyodbc.connect(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};DATABASE={db_name};UID={username};PWD={password}",
    autocommit=True
)
cursor = conn.cursor()
cursor.fast_executemany = True
print(f"Connected to {db_name} database!")


Connected to kaggleDataSet database!


In [23]:
import sqlalchemy as sal

engine = sal.create_engine(
    f"mssql+pyodbc://{username}:{password}@{server}/{db_name}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)


In [24]:
df.to_sql(
    name='df_orders',
    con=engine,
    index=False,
    if_exists='replace'   # first time
)


94

In [25]:
if_exists='append'
